# 01. Data Preparation

## 1. Data Audit & Cleaning

In [73]:
from pathlib import Path 
import pandas as pd 


In [74]:
data_path = Path("DILIrank2_Dataset_FDA.xlsx")

# first row should be skipped as it contains the title of the dataset, not the column names
data = pd.read_excel(data_path, header=1, dtype=str)


In [75]:
display(data.head(5))
print(data.shape)


,LTKBID,CompoundName,SeverityClass,LabelSection,vDILI-Concern,Comment
0,LT00040,Abacavir sulfate,8,Warnings & precautions,vMOST-DILI-concern,Unchanged
1,LT03618,Abaloparatide,0,No match,vNo-DILI-concern,New
2,LT01402,Abatacept,0,No match,vLess-DILI-concern,Unchanged
3,LT01330,Abciximab,0,No match,vNo-DILI-concern,Unchanged
4,LT03619,Abemaciclib,3,Warnings & precautions,vLess-DILI-concern,New


(1336, 6)


In [76]:
data.columns.tolist()

['LTKBID',
 'CompoundName',
 'SeverityClass',
 'LabelSection',
 'vDILI-Concern',
 'Comment']

In [77]:
# look for unique values in the following columns
cols = ["SeverityClass", "LabelSection", "vDILI-Concern", "Comment"]

for col in cols:
    print(f"Column: {col}")
    print(data[col].unique())
    print("\n")


Column: SeverityClass
<StringArray>
['8', '0', '3', '4', '5', '7', '2', '6', '1']
Length: 9, dtype: str


Column: LabelSection
<StringArray>
['Warnings & precautions',               'No match',      'Adverse reactions',
            'Box warning',              'Withdrawn',           'Discontinued']
Length: 6, dtype: str


Column: vDILI-Concern
<StringArray>
[    'vMOST-DILI-concern',       'vNo-DILI-concern',     'vLess-DILI-concern',
     'vMost-DILI-concern', 'Ambiguous-DILI-concern',       'vNo-DILI-Concern']
Length: 6, dtype: str


Column: Comment
<StringArray>
['Unchanged', 'New', 'Revised']
Length: 3, dtype: str




In [78]:
# normalize DILI concern labels
concern_mapping = {
    "vMOST-DILI-concern": "vMost-DILI-concern",
    "vMost-DILI-concern": "vMost-DILI-concern",
    "vLess-DILI-concern": "vLess-DILI-concern",
    "vNo-DILI-concern": "vNo-DILI-concern",
    "vNo-DILI-Concern": "vNo-DILI-concern",
    "Ambiguous-DILI-concern": "Ambiguous-DILI-concern"
}

data["vDILI-Concern"] = data["vDILI-Concern"].map(concern_mapping)
data["vDILI-Concern"].value_counts(dropna=False)


vDILI-Concern
vNo-DILI-concern          414
Ambiguous-DILI-concern    354
vLess-DILI-concern        351
vMost-DILI-concern        217
Name: count, dtype: int64

In [79]:
# check for rows with missing values
data.isnull().sum()


LTKBID           0
CompoundName     0
SeverityClass    0
LabelSection     0
vDILI-Concern    0
Comment          0
dtype: int64

In [80]:
# check for duplicates in the dataset

# duplicate IDs
print("Duplicate LTKBID:", data["LTKBID"].duplicated().sum())

# duplicate compound names
print("Duplicate compound names:", data["CompoundName"].duplicated().sum())


Duplicate LTKBID: 0
Duplicate compound names: 0


In [81]:
# display the first 5 rows of the dataset
display(data.head(5))

,LTKBID,CompoundName,SeverityClass,LabelSection,vDILI-Concern,Comment
0,LT00040,Abacavir sulfate,8,Warnings & precautions,vMost-DILI-concern,Unchanged
1,LT03618,Abaloparatide,0,No match,vNo-DILI-concern,New
2,LT01402,Abatacept,0,No match,vLess-DILI-concern,Unchanged
3,LT01330,Abciximab,0,No match,vNo-DILI-concern,Unchanged
4,LT03619,Abemaciclib,3,Warnings & precautions,vLess-DILI-concern,New


In [82]:
# data summary
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 1336 entries, 0 to 1335
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   LTKBID         1336 non-null   str  
 1   CompoundName   1336 non-null   str  
 2   SeverityClass  1336 non-null   str  
 3   LabelSection   1336 non-null   str  
 4   vDILI-Concern  1336 non-null   str  
 5   Comment        1336 non-null   str  
dtypes: str(6)
memory usage: 62.8 KB


## 2. Molecular Structure Retrieval